In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import find_peaks

In [ ]:
# Import CGM data from Fran
cgm = pd.read_sas(
    "/Users/tim/Library/CloudStorage/OneDrive-UW/UWMDI/Andrea Steck/Advanced CGM and OGTT/Data_Raw/Final data20250801/tim_allcgmclean_gt4days.sas7bdat",
    format="sas7bdat",
    encoding="latin-1",
)
cgm["DateTime"] = pd.to_datetime(
    cgm.newsensortime, unit="s", origin=pd.Timestamp("1960-01-01"), utc=True
).dt.tz_convert("America/Denver")

In [ ]:
# Define the plotting function
def plot_cgm_peaks(
    id,
    dov,
    df=cgm,
    sensor_time="DateTime",
    glucose="sensor_glucose",
    min_height=100,
    rise=50,
    time_between_peaks=120,
    min_time_of_peak=30,
):
    # Filter and sort data
    df=df.sort_values("DateTime").reset_index()
    df = df[df.ID == id]
    df = df[df.dov_CGM == dov]
    # Find peaks using the Ruiz et al. cutoffs
    # Sampling interval (assumes uniform spacing)
    dt = np.median(np.diff(df[sensor_time]))  # minutes per sample
    dt = dt.seconds / 60
    samples_per_2h = int(time_between_peaks / dt)  # minimum distance between peaks
    samples_per_30min = int(min_time_of_peak / dt)
    # Find peaks
    peaks, props = find_peaks(
        df[glucose],
        height=min_height,
        prominence=rise,
        distance=samples_per_2h,  # ≥ 2 h between peaks
        width=samples_per_30min,
        rel_height=1,
    )
    # Plot
    ax = sns.lineplot(data=df, x=sensor_time, y=glucose)
    for left, right in zip(props["left_bases"], props["right_bases"]):
        ax.axvspan(
            xmin=df[sensor_time].iloc[left],
            xmax=df[sensor_time].iloc[right],
            color="red",
            alpha=0.2,
        )
    print(ax)
    # Make a peak dataframe
    peak_df = pd.DataFrame({"start": df[sensor_time].iloc[props["left_bases"].tolist()].values, 
                            "end": df[sensor_time].iloc[props["right_bases"].tolist()].values,
                            "peak_glucose": df[glucose].iloc[peaks].tolist()})
    print(peak_df)


In [ ]:
plot_cgm_peaks(id="00174-0", dov="2015-01-13")